# 07 Quantum Grid And Payoff Encoding

Construct log-price grids, payoff vectors, normalized state encodings, and duplicated boundary encodings.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
n_qubits = 5
K = 24000.0
x_grid, S_grid, dx = make_log_price_grid(n_qubits, center_price=K, x_width=config["quantum"]["x_width"])
call_payoff = payoff_vector(S_grid, K, "call")
put_payoff = payoff_vector(S_grid, K, "put")
call_state, call_norm = normalize_state(call_payoff)
put_state, put_norm = normalize_state(put_payoff)
dup_put = duplicate_payoff_encoding(put_payoff)
dup_state, dup_norm = normalize_state(dup_put)
assert abs(np.linalg.norm(call_state) - 1.0) < 1e-12
assert abs(np.linalg.norm(put_state) - 1.0) < 1e-12
assert abs(np.linalg.norm(dup_state) - 1.0) < 1e-12
print("VALIDATION PASSED: payoff vector normalization")
F = qft_matrix(2 ** n_qubits)
assert np.linalg.norm(F.conjugate().T @ F - np.eye(2 ** n_qubits)) < 1e-10
assert np.linalg.norm(apply_inverse_qft_state(apply_qft_state(put_state)) - put_state) < 1e-10
print("VALIDATION PASSED: QFT unitarity and inverse QFT correctness")
plt.figure()
plt.plot(S_grid, call_payoff, label="Call payoff")
plt.plot(S_grid, put_payoff, label="Put payoff")
plt.title("Quantum grid payoff encoding")
plt.xlabel("Index level")
plt.ylabel("Payoff")
plt.legend()
save_current_figure("07_payoff_encoding.png")
save_table(pd.DataFrame({"x": x_grid, "S": S_grid, "call_payoff": call_payoff, "put_payoff": put_payoff, "put_state_real": put_state.real}), "07_quantum_grid_payoff.csv")
{"call_norm": call_norm, "put_norm": put_norm, "duplicated_norm": dup_norm, "dx": dx}
